In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


In [ ]:
price_dfs = []
for day in [-2, -1, 0]:
    df = pd.read_csv(f"data/prices_round_1_day_{day}.csv", sep=";")
    df["day"] = day
    price_dfs.append(df)
prices = pd.concat(price_dfs, ignore_index=True)
prices = prices[(prices["bid_price_1"] > 0) & (prices["ask_price_1"] > 0)].copy()
prices["mid"] = 0.5 * (prices["bid_price_1"] + prices["ask_price_1"])
prices["spread"] = prices["ask_price_1"] - prices["bid_price_1"]
prices.head()


In [ ]:
osm = prices[prices["product"] == "ASH_COATED_OSMIUM"].copy()
osm.groupby("day")["mid"].agg(["mean", "std", "min", "max"])


In [ ]:
pep = prices[prices["product"] == "INTARIAN_PEPPER_ROOT"].copy()
X = pep[["day", "timestamp"]].copy()
X["timestamp"] = X["timestamp"] / 1000.0
y = pep["mid"]
model = LinearRegression().fit(X, y)
print("intercept:", model.intercept_)
print("coef_day:", model.coef_[0])
print("coef_timestamp/1000:", model.coef_[1])
print("r2:", model.score(X, y))


In [ ]:
for product in ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]:
    sub = prices[prices["product"] == product]
    plt.figure(figsize=(12,4))
    plt.plot(sub["timestamp"], sub["mid"])
    plt.title(product)
    plt.show()


In [ ]:
raw = pd.concat(price_dfs, ignore_index=True)
raw["missing_bid"] = raw["bid_price_1"] <= 0
raw["missing_ask"] = raw["ask_price_1"] <= 0
raw.groupby("product")[["missing_bid", "missing_ask"]].mean()
